In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHTS = {
    "Dry_Green_g": 0.1,
    "Dry_Dead_g":  0.1,
    "Dry_Clover_g": 0.1,
    "GDM_g":        0.2,
    "Dry_Total_g":  0.5,
}
WEIGHT_VECTOR = np.array([WEIGHTS[t] for t in TARGETS], dtype=np.float32)

DATASET_DIR   = "/kaggle/input/competitions/csiro-biomass"
TRAIN_CSV     = os.path.join(DATASET_DIR, "train.csv")
TEST_CSV      = os.path.join(DATASET_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train")
TEST_IMG_DIR  = os.path.join(DATASET_DIR, "test")

print("train.csv   :", os.path.exists(TRAIN_CSV))
print("test.csv    :", os.path.exists(TEST_CSV))
print("train dir   :", os.path.exists(TRAIN_IMG_DIR))
print("test  dir   :", os.path.exists(TEST_IMG_DIR))


def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]
    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target",
    ).reset_index()
    y_values = df_wide[TARGETS].values
    return df_wide, y_values


def prepare_submission(test_csv_path, predictions, image_ids):
    df_test = pd.read_csv(test_csv_path)
    pred_dict = {
        img_id: {col: val for col, val in zip(TARGETS, row)}
        for img_id, row in zip(image_ids, predictions)
    }

    def get_pred(row):
        img_id = row["sample_id"].split("__")[0]
        return max(0.0, float(pred_dict.get(img_id, {}).get(row["target_name"], 0.0)))

    df_test["target"] = df_test.apply(get_pred, axis=1)
    return df_test[["sample_id", "target"]]


def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR.astype(np.float64)
    yt = y_true.reshape(-1)
    yp = y_pred.reshape(-1)
    ww = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {
        t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
        for i, t in enumerate(TARGETS)
    }

In [ ]:
class ResNet50Extractor:
    """Mirrors works_with_kaggle.ipynb: pretrained if available, else random init.
       Concatenates features from left/right halves of the 2000x1000 image -> 4096-d."""
    def __init__(self, device="cpu"):
        self.device = device
        try:
            weights = models.ResNet50_Weights.DEFAULT
            resnet = models.resnet50(weights=weights)
            print("Loaded pretrained ResNet50 weights.")
        except Exception as e:
            print("No internet -> random init fallback:", e)
            resnet = models.resnet50(weights=None)

        self.model = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
        self.preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    @torch.no_grad()
    def _one(self, pil_img):
        t = self.preprocess(pil_img).unsqueeze(0).to(self.device)
        return self.model(t).flatten().cpu().numpy().astype(np.float32)

    def get_features(self, image_path):
        if not os.path.exists(image_path):
            print("Missing:", image_path)
            return np.zeros(4096, dtype=np.float32)
        try:
            img = Image.open(image_path).convert("RGB")
            left  = img.crop((0, 0, 1000, 1000))
            right = img.crop((1000, 0, 2000, 1000))
            return np.concatenate([self._one(left), self._one(right)], axis=0).astype(np.float32)
        except Exception as e:
            print(f"Error {image_path}: {e}")
            return np.zeros(4096, dtype=np.float32)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_df, y_all = load_train_data(TRAIN_CSV)
print("Train rows:", len(train_df), " y:", y_all.shape)

extractor = ResNet50Extractor(device=device)

all_feats, all_ids = [], []
for i, row in enumerate(train_df.itertuples(index=False), 1):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row.image_id}.jpg")
    all_feats.append(extractor.get_features(img_path))
    all_ids.append(row.image_id)
    if i % 100 == 0:
        print(f"  {i}/{len(train_df)}")

X_all = np.asarray(all_feats, dtype=np.float32)
print("X_all:", X_all.shape, " y_all:", y_all.shape)

np.save("/kaggle/working/X_all.npy", X_all)
np.save("/kaggle/working/y_all.npy", y_all)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim=4096, latent_dim=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, latent_dim),
        )
    def forward(self, x): return self.net(x)


class Head(nn.Module):
    def __init__(self, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, output_dim),
        )
    def forward(self, z): return self.net(z)


class BiomassModel(nn.Module):
    def __init__(self, input_dim=4096, latent_dim=32, output_dim=5, dropout=0.1):
        super().__init__()
        self.encoder = Encoder(input_dim, latent_dim, dropout)
        self.head    = Head(latent_dim, output_dim, dropout)
    def forward(self, x): return self.head(self.encoder(x))

In [ ]:
_LOSS_W = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)

def weighted_smooth_l1(pred, target, weights, beta=1.0):
    loss = F.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss * weights.to(pred.device)).mean()


def run_training(model, X_tr, y_tr, X_val, y_val, *,
                 epochs=80, batch_size=32, lr=3e-4, weight_decay=1e-3,
                 seed=42, device="cpu", verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)

    def to_loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.float32))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

    tr_loader  = to_loader(X_tr,  y_tr,  True)
    val_loader = to_loader(X_val, y_val, False) if X_val is not None else None

    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 100)

    for epoch in range(1, epochs + 1):
        model.train()
        tr_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = weighted_smooth_l1(model(xb), yb, _LOSS_W)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * len(xb)
        tr_loss /= len(tr_loader.dataset)
        sched.step()

        if verbose and (epoch == 1 or epoch % 10 == 0):
            msg = f"  ep {epoch:3d}  tr={tr_loss:.4f}"
            if val_loader is not None:
                model.eval()
                preds, trues = [], []
                with torch.no_grad():
                    for xb, yb in val_loader:
                        preds.append(model(xb.to(device)).cpu().numpy())
                        trues.append(yb.numpy())
                preds = np.concatenate(preds); trues = np.concatenate(trues)
                msg += f"  val_R2={weighted_global_r2(trues, preds):.4f}"
            print(msg)
    return model

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)
print("Train:", X_tr.shape, " Val:", X_val.shape)

model = BiomassModel(input_dim=X_all.shape[1], latent_dim=32, output_dim=5, dropout=0.1).to(device)
run_training(model, X_tr, y_tr, X_val, y_val,
             epochs=80, batch_size=32, lr=3e-4, weight_decay=1e-3,
             seed=42, device=device)

model.eval()
with torch.no_grad():
    val_pred = model(torch.tensor(X_val, dtype=torch.float32, device=device)).cpu().numpy()
print("Validation weighted R2:", weighted_global_r2(y_val, val_pred))
print("Validation RMSE/target:", rmse_per_target(y_val, val_pred))

In [ ]:
final_model = BiomassModel(input_dim=X_all.shape[1], latent_dim=32, output_dim=5, dropout=0.1).to(device)
run_training(final_model, X_all, y_all, None, None,
             epochs=80, batch_size=32, lr=3e-4, weight_decay=1e-3,
             seed=42, device=device, verbose=True)

df_test = pd.read_csv(TEST_CSV)
df_test["image_id"] = df_test["sample_id"].str.split("__").str[0]
df_test_unique = df_test.drop_duplicates("image_id").copy()
print("Unique test images:", len(df_test_unique))

test_feats, test_ids = [], []
for i, row in enumerate(df_test_unique.itertuples(index=False), 1):
    img_path = os.path.join(TEST_IMG_DIR, f"{row.image_id}.jpg")
    test_feats.append(extractor.get_features(img_path))
    test_ids.append(row.image_id)
    if i % 100 == 0:
        print(f"  {i}/{len(df_test_unique)}")

X_test = np.asarray(test_feats, dtype=np.float32)
final_model.eval()
with torch.no_grad():
    test_pred = final_model(torch.tensor(X_test, dtype=torch.float32, device=device)).cpu().numpy()

submission = prepare_submission(TEST_CSV, test_pred, test_ids)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print(submission.head())
print("Saved /kaggle/working/submission.csv  rows:", len(submission))
print("Working dir:", os.listdir("/kaggle/working"))